# CashCompass: build `embed_model.pte`, `tokenizer.model` and `finance_assistant.pte`

Run the cells **top to bottom** (Runtime ▸ Run all works for Parts 0–3; Part 4 is separate and heavy).

| Part | Output | Time (free Colab) |
|---|---|---|
| 1 | `tokenizer.model` (SentencePiece, 32k) | ~1 min |
| 2 | `embed_model.pte` (384-dim, cosine, XNNPACK, quantized) | ~5 min |
| 3 | Check the `.pte` runs and matches PyTorch | ~1 min |
| 4 | `finance_assistant.pte` (1B LLM, 8da4w) | 20-40 min, needs ≥ 12 GB RAM |
| 5 | Zip + download | seconds |

> ExecuTorch changes quickly. If a cell fails with an import error, the cell prints a hint; check <https://docs.pytorch.org/executorch/stable/> for the current API.

## Part 0: Install

In [ ]:
!pip -q install executorch transformers sentence-transformers sentencepiece datasets huggingface_hub
# If Colab shows "Restart session", click it, then run from the next cell (do not re-run this one).

In [ ]:
import os, shutil, time, torch, numpy as np
OUT = "/content/cashcompass_assets"
os.makedirs(f"{OUT}/models", exist_ok=True)
print("torch", torch.__version__)
import executorch; print("executorch", getattr(executorch, "__version__", "?"))

## Part 1: `tokenizer.model`

**Important:** a tokenizer must match the model that uses it.
- For the **LLM**, use the *base model's own* `tokenizer.model` (Option A). TinyLlama already has a 32k SentencePiece vocab, so it fits your spec. A freshly trained SPM would give wrong token IDs and break a pretrained LLM.
- Option B trains a brand-new 32k SPM with finance terms. Only use it if you train a model **from scratch** with it.
- The **embedder** (bge-small) uses a BERT WordPiece tokenizer (`vocab.txt`), not SentencePiece. Part 2 exports that file too; your Kotlin `Tokenizer` needs to handle both.

In [ ]:
# ---- Option A (recommended): take the LLM's own tokenizer.model ----
from huggingface_hub import hf_hub_download
import sentencepiece as spm

LLM_REPO = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"   # change to your base model
p = hf_hub_download(LLM_REPO, "tokenizer.model")
shutil.copy(p, f"{OUT}/models/tokenizer.model")

sp = spm.SentencePieceProcessor(model_file=f"{OUT}/models/tokenizer.model")
print("vocab size:", sp.vocab_size(), "| bos:", sp.bos_id(), "| eos:", sp.eos_id())
for t in ["₹1,250 paid via UPI to Swiggy", "NEFT credit of Rs 45,000 salary", "EMI debited for loan, SIP installment"]:
    ids = sp.encode(t); print(len(ids), sp.decode(ids) == t, sp.id_to_piece(ids)[:8])

## Part 2: `embed_model.pte` (merchant → category embedder)

Exports `BAAI/bge-small-en-v1.5` (384-dim) with CLS pooling + L2 normalisation baked in, so the app only needs cosine/dot product.

**Optional fine-tune first:** if you have a CSV with `merchant,category` columns, upload it as `/content/merchants.csv` and set `FINE_TUNE = True`.

**Size note (measured):** with this recipe the `.pte` is about **69 MB** for a bge-small-sized model (tested). The 12 transformer layers become INT8, but the 30,522 x 384 word-embedding table stays full precision (~47 MB). That does **not** meet the ≤ 15 MB budget in your spec. Realistic ways to get there, to decide with M2:
1. **Trim the vocabulary** to the few thousand tokens your merchant strings actually use (the embedding table shrinks proportionally; needs a custom tokenizer).
2. Quantize the embedding table too (torchao). I tried this and the ExecuTorch lowering step failed with a missing-op error, so it needs extra custom-op setup; check the ExecuTorch docs for your version.
3. Raise the budget to ~70 MB; for an app that already ships a ~700 MB LLM this is minor.

In [ ]:
EMBED_MODEL = "BAAI/bge-small-en-v1.5"
MAX_LEN = 64            # merchant strings are short; static shape keeps XNNPACK fast
FINE_TUNE = False       # True = fine-tune on /content/merchants.csv (merchant,category)

if FINE_TUNE:
    import pandas as pd
    from sentence_transformers import SentenceTransformer, InputExample, losses
    from torch.utils.data import DataLoader
    df = pd.read_csv("/content/merchants.csv")
    st = SentenceTransformer(EMBED_MODEL)
    ex = [InputExample(texts=[r.merchant, r.category]) for r in df.itertuples()]
    dl = DataLoader(ex, shuffle=True, batch_size=32)
    st.fit(train_objectives=[(dl, losses.MultipleNegativesRankingLoss(st))], epochs=1, warmup_steps=20)
    st.save("/content/embed_ft"); EMBED_MODEL = "/content/embed_ft"

In [ ]:
from transformers import AutoTokenizer, AutoModel

tok = AutoTokenizer.from_pretrained(EMBED_MODEL)
base = AutoModel.from_pretrained(EMBED_MODEL, attn_implementation="eager").eval()

class Embedder(torch.nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    def forward(self, input_ids, attention_mask):
        # build the additive mask ourselves: keeps the exported graph simple (tested with torch 2.15 / transformers 5.x)
        ext = (1.0 - attention_mask[:, None, None, :].to(torch.float32)) * -10000.0
        h = self.m.embeddings(input_ids=input_ids)
        h = self.m.encoder(h, attention_mask=ext)
        h = (h.last_hidden_state if hasattr(h, "last_hidden_state") else h[0])[:, 0]   # CLS pooling
        return torch.nn.functional.normalize(h, dim=-1)

model = Embedder(base).eval()
enc = tok(["Swiggy"], padding="max_length", max_length=MAX_LEN, truncation=True, return_tensors="pt")
example = (enc["input_ids"], enc["attention_mask"])
with torch.no_grad():
    ref = model(*example)
print("output shape:", tuple(ref.shape))   # expect (1, 384)

In [ ]:
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower

QUANTIZE = True   # dynamic INT8 weights/activations via XNNPACK

if QUANTIZE:
    from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import XNNPACKQuantizer, get_symmetric_quantization_config
    try:
        from torchao.quantization.pt2e.quantize_pt2e import prepare_pt2e, convert_pt2e
    except ImportError:
        from torch.ao.quantization.quantize_pt2e import prepare_pt2e, convert_pt2e
    cfg = get_symmetric_quantization_config(is_per_channel=True, is_dynamic=True)
    quantizer = XNNPACKQuantizer().set_global(cfg)
    if hasattr(torch.export, "export_for_training"):          # torch 2.5 - 2.9
        gm = torch.export.export_for_training(model, example).module()
    else:                                                      # newer torch: export() is already the training IR
        gm = torch.export.export(model, example).module()
    gm = prepare_pt2e(gm, quantizer)
    with torch.no_grad():
        gm(*example)                       # calibration (dynamic quant needs no dataset)
    gm = convert_pt2e(gm)
    exported = torch.export.export(gm, example)
else:
    exported = torch.export.export(model, example)

et_prog = to_edge_transform_and_lower(exported, partitioner=[XnnpackPartitioner()]).to_executorch()
EMBED_PTE = f"{OUT}/models/embed_model.pte"
with open(EMBED_PTE, "wb") as f:
    et_prog.write_to_file(f)
print(f"embed_model.pte = {os.path.getsize(EMBED_PTE)/1e6:.1f} MB")

# the embedder's tokenizer files (your Kotlin side needs vocab.txt)
tok.save_pretrained(f"{OUT}/models/embed_tokenizer")

## Part 3: Verify the `.pte` (runs + matches PyTorch)

In [ ]:
from executorch.runtime import Runtime
prog = Runtime.get().load_program(EMBED_PTE)
method = prog.load_method("forward")
out = method.execute(list(example))[0]
cos = torch.nn.functional.cosine_similarity(out, ref).item()
print("cosine(pte, pytorch fp32) =", round(cos, 4), "(want > 0.98)")

t = time.time()
for _ in range(30): method.execute(list(example))
print("latency on Colab CPU: %.1f ms/inference (phone numbers will differ; measure on a real device)" % ((time.time()-t)/30*1000))

# quick sanity: similar merchants should be closer than unrelated ones
def emb(txt):
    e = tok([txt], padding="max_length", max_length=MAX_LEN, truncation=True, return_tensors="pt")
    return method.execute([e["input_ids"], e["attention_mask"]])[0][0]
a, b, c = emb("Swiggy food order"), emb("Zomato food delivery"), emb("Airtel postpaid bill")
print("swiggy~zomato", float(a @ b), "| swiggy~airtel", float(a @ c))

## Part 4: `finance_assistant.pte` (LLM, 8-bit activations + 4-bit weights)

This uses **optimum-executorch** (Hugging Face model → ExecuTorch `.pte` for XNNPACK).
- Needs a **High-RAM** Colab runtime (Runtime ▸ Change runtime type). A GPU is not needed.
- It may reinstall `torch`, so after this cell finishes installing, restart the session and run only the cells below it.
- This is the `.pte` of the **base chat model**. To use your QLoRA fine-tune, merge the adapter into the base weights first (`peft: model.merge_and_unload().save_pretrained("/content/merged")`) and set `LLM_SRC = "/content/merged"`.
- Flags below are the common ones at time of writing; if the CLI rejects one, run `optimum-cli export executorch --help`.

In [ ]:
RUN_LLM_EXPORT = False     # set True when you are ready (long + RAM heavy)
LLM_SRC = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

if RUN_LLM_EXPORT:
    !pip -q install optimum-executorch
    !optimum-cli export executorch \
        --model {LLM_SRC} \
        --task text-generation \
        --recipe xnnpack \
        --use_custom_sdpa --use_custom_kv_cache \
        --qlinear 8da4w --qembedding 8w \
        --output_dir /content/llm_out
    !ls -la /content/llm_out
    shutil.copy("/content/llm_out/model.pte", f"{OUT}/models/finance_assistant.pte")
    print("finance_assistant.pte =", os.path.getsize(f"{OUT}/models/finance_assistant.pte")/1e6, "MB")

## Part 5: Zip and download

In [ ]:
# manifest with SHA-256 so your Gradle copy task can verify files
import hashlib, json, glob
man = {}
for p in glob.glob(f"{OUT}/models/**/*", recursive=True):
    if os.path.isfile(p):
        man[os.path.relpath(p, OUT)] = {"bytes": os.path.getsize(p), "sha256": hashlib.sha256(open(p,"rb").read()).hexdigest()}
json.dump(man, open(f"{OUT}/manifest.json", "w"), indent=2)
print(json.dumps(man, indent=2))

shutil.make_archive("/content/cashcompass_assets", "zip", OUT)
from google.colab import files
files.download("/content/cashcompass_assets.zip")
# Or save to Drive instead:
# from google.colab import drive; drive.mount('/content/drive'); shutil.copy("/content/cashcompass_assets.zip", "/content/drive/MyDrive/")

## Where the files go in the project
```
ml/assets/embed_model.pte           → copy from models/embed_model.pte
ml/assets/finance_assistant.pte     → copy from models/finance_assistant.pte
ml/assets/tokenizer.model           → copy from models/tokenizer.model
ml/assets/embed_tokenizer/vocab.txt → embedder WordPiece vocab (for Kotlin)
```
Then commit with Git LFS; M3's Gradle task copies them into `android/app/src/main/assets/`.